# Solution PromoPro — étape 2A : évaluation d'un modèle déjà entraîné

Modèle testé : `Yytsi/floorplan-to-3d-walls` (U-Net, encodeur ResNet-34, 4 classes : sol / mur / porte / fenêtre), entraîné sur **CubiCasa5K**.

⚠️ **Licence** : le modèle est publié sous MIT, mais CubiCasa5K est **CC BY-NC 4.0** (usage non commercial). Ce carnet sert **uniquement à évaluer** la qualité atteignable ; ce modèle ne peut pas être mis en production dans PromoPro sans licence commerciale CubiCasa. Voir IA-INTERNE.md.

Ce carnet est prévu pour **Google Colab** (niveau gratuit, CPU suffisant pour une inférence). Exécutez les cellules dans l'ordre. À la fin, vous téléchargez le masque et un JSON au même format que celui rendu par Gemini, pour comparer.

In [ ]:
# 1. Dépendances (2 à 3 minutes)
%pip -q install segmentation-models-pytorch huggingface_hub safetensors opencv-python-headless pyyaml

In [ ]:
# 2. Téléchargement des poids et de la configuration
from huggingface_hub import hf_hub_download
import yaml, json
poids = hf_hub_download('Yytsi/floorplan-to-3d-walls', 'best.safetensors')
config = yaml.safe_load(open(hf_hub_download('Yytsi/floorplan-to-3d-walls', 'config.yaml')))
print(json.dumps(config, indent=2)[:3000])  # à lire : nom de l'encodeur, nombre de classes, taille d'image, normalisation

In [ ]:
# 3. Construction du modèle et chargement des poids
import torch, segmentation_models_pytorch as smp
from safetensors.torch import load_file

def lire(cle, defaut):
    # la configuration publiée peut nommer les champs différemment : on cherche la clé à tous les niveaux
    pile = [config]
    while pile:
        d = pile.pop()
        if isinstance(d, dict):
            if cle in d: return d[cle]
            pile.extend(d.values())
    return defaut

def entier(v, defaut):
    # la valeur peut être un nombre, une liste [hauteur, largeur], une liste de noms de classes ou une chaîne
    if isinstance(v, (list, tuple)):
        if v and all(isinstance(x, (int, float)) for x in v): return int(max(v))
        return len(v) if v else defaut
    if isinstance(v, dict): return len(v) or defaut
    try: return int(v)
    except (TypeError, ValueError): return defaut

encodeur = str(lire('encoder_name', lire('encoder', 'resnet34')))
classes = entier(lire('classes', lire('num_classes', lire('class_names', 4))), 4)
taille = entier(lire('image_size', lire('img_size', lire('input_size', 512))), 512)
print('encodeur', encodeur, '| classes', classes, '| taille', taille)

modele = smp.Unet(encoder_name=encodeur, encoder_weights=None, classes=classes)
etat = load_file(poids)
# Alignement automatique des préfixes de clés (ex. 'model.encoder…' → 'encoder…')
attendues = set(modele.state_dict().keys())
if not attendues & set(etat.keys()):
    for prefixe in ('model.', 'module.', 'net.', 'unet.'):
        candidat = {k[len(prefixe):]: v for k, v in etat.items() if k.startswith(prefixe)}
        if attendues & set(candidat.keys()):
            etat = candidat; print('préfixe retiré :', prefixe); break
manquants, inattendus = modele.load_state_dict(etat, strict=False)
print('clés manquantes :', len(manquants), '| inattendues :', len(inattendus))
if manquants or inattendus:
    print('→ Premières clés du fichier :', list(etat)[:5])
    print('→ Premières clés attendues :', list(attendues)[:5])
assert len(manquants) < len(attendues) // 2, 'Trop de clés manquantes : les poids ne correspondent pas à cette architecture.'
modele.eval()

In [ ]:
# 4. Déposez l'image du plan (villa_plan_2d.png) quand la boîte de dialogue s'ouvre
from google.colab import files
depose = files.upload()
chemin = next(iter(depose))
print('plan :', chemin)

In [ ]:
# 5. Inférence : masque des 4 classes (0 sol, 1 mur, 2 porte, 3 fenêtre)
# Le modèle a été entraîné en « letterbox » (config.yaml) : on garde les proportions du plan et on complète
# en blanc jusqu'au carré, au lieu d'écraser l'image. Les plans larges sont aussi passés par tuiles si le
# côté dépasse 2 × la taille d'entraînement, pour ne pas perdre les murs fins.
import cv2, numpy as np
img = cv2.cvtColor(cv2.imread(chemin), cv2.COLOR_BGR2RGB)
H, W = img.shape[:2]
cote = (taille // 32) * 32
moy, ecart = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

def predire(bloc):
    h, w = bloc.shape[:2]; ech = cote / max(h, w)
    redim = cv2.resize(bloc, (max(1, round(w * ech)), max(1, round(h * ech))), interpolation=cv2.INTER_AREA)
    toile = np.full((cote, cote, 3), 255, np.uint8); toile[:redim.shape[0], :redim.shape[1]] = redim
    x = torch.from_numpy(((toile.astype(np.float32) / 255 - moy) / ecart).transpose(2, 0, 1)).float()[None]
    with torch.no_grad(): m = modele(x).argmax(1)[0].numpy().astype(np.uint8)
    return cv2.resize(m[:redim.shape[0], :redim.shape[1]], (w, h), interpolation=cv2.INTER_NEAREST)

if max(H, W) <= 2 * cote:
    masque = predire(img)
else:  # tuiles de 1024 px avec recouvrement, vote par le dernier écrit
    pas = 2 * cote; rec = 128; masque = np.zeros((H, W), np.uint8)
    for y in range(0, H, pas - rec):
        for x in range(0, W, pas - rec):
            y2, x2 = min(H, y + pas), min(W, x + pas); masque[y:y2, x:x2] = predire(img[y:y2, x:x2])
print('pixels par classe :', {c: int((masque == c).sum()) for c in range(classes)})

couleurs = np.array([[255, 255, 255], [20, 20, 20], [230, 120, 0], [0, 120, 230]], dtype=np.uint8)
superposition = (0.55 * img + 0.45 * couleurs[np.clip(masque, 0, 3)]).astype(np.uint8)
cv2.imwrite('masque.png', cv2.cvtColor(superposition, cv2.COLOR_RGB2BGR))
# Masque seul, en couleurs franches, pour juger murs (noir), portes (orange) et fenêtres (bleu) sans le plan dessous
cv2.imwrite('masque_seul.png', cv2.cvtColor(couleurs[np.clip(masque, 0, 3)], cv2.COLOR_RGB2BGR))
from IPython.display import Image, display
display(Image('masque.png', width=700)); display(Image('masque_seul.png', width=700))

In [ ]:
# 6. Pièces = composantes connexes du sol, les murs ET les portes/fenêtres faisant barrière (une porte ouverte
# ne doit pas fusionner deux pièces) ; portes = composantes de la classe porte de taille plausible.
# Résultat au même format que le JSON Gemini (coordonnées relatives), sans nom de pièce : ce modèle ne lit pas le texte.
barriere = (masque != 0).astype(np.uint8)
barriere = cv2.dilate(barriere, np.ones((7, 7), np.uint8))  # referme les petites coupures des murs
sol = (1 - barriere).astype(np.uint8)
n, etiquettes, stats, _ = cv2.connectedComponentsWithStats(sol, connectivity=4)
seuil = 0.004 * H * W  # ignore les miettes (< 0,4 % de l'image)
pieces = []
for i in range(1, n):
    x0, y0, w, h, aire = stats[i]
    if aire < seuil or w >= 0.98 * W or (x0 == 0 or y0 == 0 or x0 + w == W or y0 + h == H):  # le fond extérieur touche le bord
        continue
    pieces.append({'nom': f'Pièce {len(pieces) + 1}', 'x': round(x0 / W, 3), 'y': round(y0 / H, 3), 'largeur': round(w / W, 3), 'hauteur': round(h / H, 3)})
porte = (masque == 2).astype(np.uint8)
n2, _, stats2, centres = cv2.connectedComponentsWithStats(porte, connectivity=8)
portes = []
for i in range(1, n2):
    x0, y0, w, h, aire = stats2[i]
    # une porte réelle est compacte (environ 0,6 à 1,2 m) et touche un mur ; le texte donne des taches allongées et isolées
    if aire < 0.0002 * H * W or max(w, h) > 0.15 * max(H, W): continue
    voisinage = masque[max(0, y0 - 6):y0 + h + 6, max(0, x0 - 6):x0 + w + 6]
    if (voisinage == 1).sum() < 0.05 * voisinage.size: continue  # pas de mur autour : probablement du texte
    portes.append({'x': round(centres[i][0] / W, 3), 'y': round(centres[i][1] / H, 3), 'mur': 'vertical' if h > w else 'horizontal'})
resultat = {'pieces': pieces, 'portes': portes, 'remarques': 'modèle Yytsi/floorplan-to-3d-walls (évaluation, licence CubiCasa5K non commerciale)'}
print(json.dumps(resultat, ensure_ascii=False, indent=2))
json.dump(resultat, open('villa_plan_2d.modele.json', 'w'), ensure_ascii=False, indent=2)
files.download('villa_plan_2d.modele.json'); files.download('masque.png'); files.download('masque_seul.png')

## 7. Comparer avec Gemini

⚠️ Déposez ici **`villa_plan_2d.gemini.json`** (9 pièces, rendu par `scripts/prototypes/test-lecture-plan.ts`, dans `scripts/prototypes/plans/reels/`), **pas** le fichier `villa_plan_2d.modele.json` produit à l'étape 6 : comparer le modèle à lui-même donne un IoU de 1,00 partout, sans valeur. La cellule refuse un fichier dont les remarques mentionnent le modèle Yytsi.

Pour chaque pièce Gemini, la pièce du modèle qui la recouvre le mieux (IoU des rectangles) est indiquée. Un IoU ≥ 0,7 signifie une pièce bien retrouvée ; une même pièce du modèle citée pour plusieurs pièces Gemini signale une fusion.

In [ ]:
gem = json.load(open(next(iter(files.upload()))))
assert 'Yytsi' not in str(gem.get('remarques', '')), 'Ce fichier est la sortie du modèle (étape 6) : déposez villa_plan_2d.gemini.json.'
def iou(a, b):
    ax1, ay1, ax2, ay2 = a['x'], a['y'], a['x'] + a['largeur'], a['y'] + a['hauteur']
    bx1, by1, bx2, by2 = b['x'], b['y'], b['x'] + b['largeur'], b['y'] + b['hauteur']
    inter = max(0, min(ax2, bx2) - max(ax1, bx1)) * max(0, min(ay2, by2) - max(ay1, by1))
    union = a['largeur'] * a['hauteur'] + b['largeur'] * b['hauteur'] - inter
    return inter / union if union else 0
print(f"{'pièce Gemini':<18}{'meilleure pièce modèle':<24}{'IoU':>6}")
retrouvees = 0
for p in gem['pieces']:
    meilleur = max(pieces, key=lambda q: iou(p, q), default=None)
    score = iou(p, meilleur) if meilleur else 0; retrouvees += score >= 0.7
    print(f"{p['nom']:<18}{(meilleur['nom'] if meilleur else '—'):<24}{score:>6.2f}")
print(f"
pièces retrouvées (IoU ≥ 0,7) : {retrouvees}/{len(gem['pieces'])} — pièces du modèle : {len(pieces)}")
print(f"portes : Gemini {len(gem['portes'])}, modèle {len(portes)}")